<a href="https://colab.research.google.com/github/kasiranaweera/CDAZZDEV-MLE-KASI/blob/main/task3_agentic/CDAZZDEV_Task3_Agentic_Workflows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3: Multi-Agent Financial Research System with Memory & Observability

Step 1 — Install Dependencies

In [1]:
# STEP 1 — Install Dependencies
!pip install -q langchain langchain-groq langgraph langchain-community
!pip install -q langsmith
!pip install -q yfinance duckduckgo-search
!pip install -q pydantic pandas numpy scipy
!pip install -q streamlit plotly
!pip install -q pyngrok

print("✅ Packages installed.")

✅ Packages installed.


In [2]:
# Suppress DeprecationWarnings — must run before any other imports
import sys
from datetime import datetime as _dt, timezone as _tz

try:
    import jupyter_client.session as _jcs

    class _PatchedDatetime(_dt):
        """Drop-in datetime subclass: utcnow() uses now(UTC) — no DeprecationWarning."""
        @classmethod
        def utcnow(cls):
            # Returns naive UTC datetime so .replace(tzinfo=utc) still works downstream
            return _dt.now(_tz.utc).replace(tzinfo=None)

    _jcs.datetime = _PatchedDatetime          # replace in the module's own namespace
    print("✅ jupyter_client datetime patched — DeprecationWarning silenced.")
except Exception as _e:
    print(f"  Patch skipped ({_e}) — warnings may still appear but are harmless.")

# Also silence any remaining DeprecationWarnings from other libraries
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

✅ jupyter_client datetime patched — DeprecationWarning silenced.


Step 2 — Imports, API Keys & Configuration

In [3]:
# STEP 2 — Imports, API Keys & Configuration
import os, json, time, uuid, operator, warnings, subprocess
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional, Annotated, Sequence, TypedDict

import pandas as pd
import numpy as np
import yfinance as yf
from pydantic import BaseModel, Field, ValidationError

warnings.filterwarnings("ignore")

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_core.messages import (
    BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage,
)
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent

# ---- Groq API Key ----
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY") or ""
except Exception:
    GROQ_API_KEY = ""
if not GROQ_API_KEY:
    GROQ_API_KEY = input("🔑 Groq API key (free at console.groq.com): ").strip()
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ---- LangSmith API Key ----
# Free account at: https://smith.langchain.com
# Add LANGSMITH_API_KEY to Colab Secrets, or leave blank to skip
try:
    from google.colab import userdata as _ud
    LANGSMITH_API_KEY = _ud.get("LANGSMITH_API_KEY") or ""
except Exception:
    LANGSMITH_API_KEY = ""
if not LANGSMITH_API_KEY:
    LANGSMITH_API_KEY = input(
        "🔑 LangSmith API key (free at smith.langchain.com — press Enter to skip): "
    ).strip()

LANGSMITH_ENABLED = bool(LANGSMITH_API_KEY)
if LANGSMITH_ENABLED:
    os.environ["LANGCHAIN_TRACING_V2"]  = "true"
    os.environ["LANGCHAIN_API_KEY"]     = LANGSMITH_API_KEY
    os.environ["LANGCHAIN_PROJECT"]     = "cdazzdev-task3-financial-research"
    os.environ["LANGCHAIN_ENDPOINT"]    = "https://api.smith.langchain.com"
    print("✅ LangSmith tracing ENABLED → project: cdazzdev-task3-financial-research")
else:
    print("ℹ️  LangSmith skipped — using local JSONL tracing only")

# ---- Constants ----
LLM_MODEL      = "llama-3.3-70b-versatile"
FALLBACK_MODEL = "openai/gpt-4o-mini"
TRACE_FILE     = "agent_trace.jsonl"
CACHE_DIR      = Path("research_cache")
DASHBOARD_FILE = Path("dashboard.py")
CACHE_DIR.mkdir(exist_ok=True)

DEFAULT_TICKER = "NVDA"   # ← change this to any ticker

print(f"\n✅ Configuration complete")
print(f"   Model        : {LLM_MODEL}")
print(f"   Ticker       : {DEFAULT_TICKER}")
print(f"   LangSmith    : {'ON' if LANGSMITH_ENABLED else 'OFF (local JSONL only)'}")

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ LangSmith tracing ENABLED → project: cdazzdev-task3-financial-research

✅ Configuration complete
   Model        : llama-3.3-70b-versatile
   Ticker       : NVDA
   LangSmith    : ON


Step 3 — Pydantic Schemas

In [4]:
# STEP 3 — Pydantic Schemas
class HeadlineSentiment(BaseModel):
    headline: str
    sentiment: str          # positive | negative | neutral
    confidence: float = Field(ge=0.0, le=1.0)
    brief_reason: str

class SentimentResult(BaseModel):
    analyses: list[HeadlineSentiment]
    aggregate_score: float = Field(ge=-1.0, le=1.0)
    overall_sentiment: str

class RiskItem(BaseModel):
    risk: str
    supporting_evidence: str
    severity: str           # high | medium | low

class HedgeStrategy(BaseModel):
    strategy: str
    rationale: str
    instruments: list[str]

class DataBrief(BaseModel):
    """Typed handoff schema — Agent A → Agent B."""
    ticker: str
    price_snapshot: dict
    volatility_metrics: dict
    sentiment_result: dict
    momentum_signal: str
    key_observations: list[str] = Field(default_factory=list)
    generated_at: str

class FinalReport(BaseModel):
    ticker: str
    financial_health_summary: str
    top_three_risks: list[RiskItem]
    hedge_strategy: HedgeStrategy
    sources_consulted: list[str]
    generated_at: str

print("✅ Pydantic schemas ready.")

✅ Pydantic schemas ready.


Step 4 — Observability: AgentTracer + Session Memory

In [5]:
# STEP 4 — Observability: AgentTracer + Session Memory
# AI-ASSISTED: Claude (claude-sonnet-4-6),
# Prompt: 'Design a JSONL observability tracer with session_id, timestamps,
#          tool inputs, truncated outputs, and wall-clock durations',
# Date: 2026-05-11

_session_memory: dict[str, Any] = {}   # short-term in-process cache

class AgentTracer:
    """
    Writes one JSON line per tool call to agent_trace.jsonl.
    Required fields: session_id, timestamp, tool_name, inputs,
                     output_truncated (≤200 chars), duration_seconds.
    """
    def __init__(self, path: str = TRACE_FILE):
        self.path = path
        self.session_id = str(uuid.uuid4())[:8]
        self._records: list[dict] = []

    def log(self, tool_name: str, inputs: dict, output: Any, duration: float) -> dict:
        record = {
            "session_id"       : self.session_id,
            "timestamp"        : datetime.now(timezone.utc).isoformat(),
            "tool_name"        : tool_name,
            "inputs"           : inputs,
            "output_truncated" : str(output)[:200],
            "duration_seconds" : round(duration, 4),
        }
        with open(self.path, "a") as fh:
            fh.write(json.dumps(record) + "\n")
        self._records.append(record)
        print(f"  📋 [{tool_name}] {duration:.2f}s → {str(output)[:55]}...")
        return record

    @property
    def records(self) -> list[dict]:
        return self._records

tracer = AgentTracer()
print(f"✅ AgentTracer active — session: {tracer.session_id}")
print(f"   Writing to: {TRACE_FILE}")

✅ AgentTracer active — session: b8d6384e
   Writing to: agent_trace.jsonl


Step 5 — Technical Indicator Helpers (no TA-Lib)

In [6]:
# STEP 5 — Technical Indicator Helpers (no TA-Lib)
def _sma(s: pd.Series, w: int) -> pd.Series:
    return s.rolling(w).mean()

def _rsi(s: pd.Series, p: int = 14) -> pd.Series:
    d  = s.diff()
    g  = d.clip(lower=0)
    l  = -d.clip(upper=0)
    ag = g.ewm(com=p - 1, min_periods=p).mean()
    al = l.ewm(com=p - 1, min_periods=p).mean()
    return 100 - 100 / (1 + ag / al.replace(0, np.nan))

def _macd(s: pd.Series, fast=12, slow=26, sig=9):
    ef = s.ewm(span=fast, adjust=False).mean()
    es = s.ewm(span=slow, adjust=False).mean()
    m  = ef - es
    sg = m.ewm(span=sig, adjust=False).mean()
    return m, sg, m - sg

def _bollinger(s: pd.Series, w=20, sd=2):
    mu = s.rolling(w).mean()
    st = s.rolling(w).std()
    return mu + sd * st, mu, mu - sd * st

def _f(v) -> Optional[float]:
    try:
        fv = float(v)
        return None if (np.isnan(fv) or np.isinf(fv)) else round(fv, 4)
    except Exception:
        return None

print("✅ Indicator helpers ready.")

✅ Indicator helpers ready.


Step 6 — Five Tool Definitions

In [7]:
# STEP 6 — Five Tool Definitions
# AI-ASSISTED: Claude (claude-sonnet-4-6),
# Prompt: 'Implement five LangChain @tool wrappers for financial research:
#          get_price_data, get_news, calculate_volatility, llm_sentiment,
#          web_search — with session memory and JSONL observability tracing',
# Date: 2026-05-11

@tool
def get_price_data(ticker: str, period: str = "2y") -> str:
    """
    Fetches OHLCV data for a stock ticker and computes five technical indicators:
    SMA-50, SMA-200, RSI-14, MACD(12,26,9), Bollinger Bands(20,2).
    Also returns current price, 52-week range, YTD return, PE ratio, and momentum signal.

    Args:
        ticker: Stock ticker symbol (e.g. 'AAPL').
        period: yfinance period string — default '2y' for two years.
    """
    start = time.time()
    try:
        stk   = yf.Ticker(ticker)
        df    = stk.history(period=period)
        if df.empty:
            raise ValueError(f"yfinance returned no data for '{ticker}'")

        close = df["Close"]
        info  = stk.info or {}

        sma50            = _sma(close, 50)
        sma200           = _sma(close, 200)
        rsi14            = _rsi(close, 14)
        macd_l, macd_s, macd_h = _macd(close)
        bb_u, bb_m, bb_l       = _bollinger(close)

        cur  = float(close.iloc[-1])
        h52  = float(close.tail(252).max())
        l52  = float(close.tail(252).min())

        yr_mask   = close.index.year == datetime.now().year
        ytd_start = float(close[yr_mask].iloc[0]) if yr_mask.any() else float(close.iloc[0])
        ytd_ret   = round((cur - ytd_start) / ytd_start * 100, 2)

        s50  = _f(sma50.iloc[-1]);   s200 = _f(sma200.iloc[-1])
        rv   = _f(rsi14.iloc[-1]) or 50.0
        mv   = _f(macd_l.iloc[-1]) or 0.0

        momentum = (
            "bullish" if (rv > 55 and mv > 0) else
            "bearish" if (rv < 45 and mv < 0) else "neutral"
        )

        result = {
            "ticker": ticker, "current_price": round(cur, 2),
            "52w_high": round(h52, 2), "52w_low": round(l52, 2),
            "ytd_return_pct": ytd_ret,
            "pe_ratio": _f(info.get("trailingPE") or info.get("forwardPE")),
            "sma_50": s50, "sma_200": s200, "rsi_14": round(rv, 2),
            "macd_line": _f(macd_l.iloc[-1]), "macd_signal": _f(macd_s.iloc[-1]),
            "macd_histogram": _f(macd_h.iloc[-1]),
            "bb_upper": _f(bb_u.iloc[-1]), "bb_lower": _f(bb_l.iloc[-1]),
            "price_vs_sma50":  "above" if s50  and cur > s50  else "below",
            "price_vs_sma200": "above" if s200 and cur > s200 else "below",
            "momentum_signal": momentum, "data_rows": len(df),
        }
        _session_memory[f"price_{ticker}"] = result
        tracer.log("get_price_data", {"ticker": ticker, "period": period},
                   result, time.time() - start)
        return json.dumps(result)

    except Exception as exc:
        err = {"error": str(exc), "ticker": ticker}
        tracer.log("get_price_data", {"ticker": ticker}, err, time.time() - start)
        return json.dumps(err)


@tool
def get_news(ticker: str, n: int = 10) -> str:
    """
    Retrieves recent news headlines for a ticker via Yahoo Finance.
    Returns JSON with headline text, source name, and publication date.

    Args:
        ticker: Stock ticker symbol.
        n: Number of headlines (default 10).
    """
    start = time.time()
    try:
        stk = yf.Ticker(ticker)
        raw = stk.news or []
        headlines = []
        for item in raw[:n]:
            content  = item.get("content", {})
            title    = content.get("title") or item.get("title") or "Untitled"
            pub      = content.get("pubDate") or item.get("providerPublishTime", "")
            provider = content.get("provider", {})
            source   = (
                provider.get("displayName", "Unknown")
                if isinstance(provider, dict) else item.get("publisher", "Unknown")
            )
            headlines.append({"headline": title, "source": source,
                               "published": str(pub)[:10]})
        if not headlines:
            headlines = [{"headline": f"No recent news for {ticker}",
                          "source": "N/A", "published": "N/A"}]
        result = {"ticker": ticker, "headlines": headlines, "count": len(headlines)}
        _session_memory[f"news_{ticker}"] = result
        tracer.log("get_news", {"ticker": ticker, "n": n}, result, time.time() - start)
        return json.dumps(result)
    except Exception as exc:
        err = {"error": str(exc), "ticker": ticker}
        tracer.log("get_news", {"ticker": ticker}, err, time.time() - start)
        return json.dumps(err)


@tool
def calculate_volatility(ticker: str, window: int | str = 30) -> str:
    """
    Computes annualised historical volatility using log-returns.
    Classifies vol regime (high/normal/low) and estimates beta vs SPY.
    Reads cached price data from session memory — demonstrates short-term memory reuse.

    Args:
        ticker: Stock ticker symbol.
        window: Rolling window in trading days as an integer (default 30, e.g. 30 or 90).
    """
    start = time.time()
    try:
        window = int(window)  # coerce to int in case LLM passes a string (e.g. "90")
        stk = yf.Ticker(ticker)
        df  = stk.history(period="1y")
        if df.empty:
            raise ValueError(f"No data for {ticker}")

        log_ret  = np.log(df["Close"] / df["Close"].shift(1)).dropna()
        roll_vol = log_ret.rolling(window).std() * np.sqrt(252)
        cur_vol  = float(roll_vol.iloc[-1])
        avg_vol  = float(roll_vol.mean())
        regime   = ("high"   if cur_vol > avg_vol * 1.2 else
                    "low"    if cur_vol < avg_vol * 0.8 else "normal")

        beta = None
        try:
            spy_df  = yf.Ticker("SPY").history(period="1y")
            spy_ret = np.log(spy_df["Close"] / spy_df["Close"].shift(1)).dropna()
            common  = log_ret.index.intersection(spy_ret.index)
            if len(common) > 30:
                cov  = np.cov(log_ret[common], spy_ret[common])
                beta = round(float(cov[0, 1] / cov[1, 1]), 3)
        except Exception:
            pass

        cached = _session_memory.get(f"price_{ticker}")
        result = {
            "ticker": ticker, "window_days": window,
            "annual_vol_pct": round(cur_vol * 100, 2),
            "avg_annual_vol_pct": round(avg_vol * 100, 2),
            "vol_regime": regime,
            "max_daily_loss_pct": round(float(log_ret.min()) * 100, 2),
            "beta_vs_spy": beta,
            "price_from_session_cache": cached.get("current_price") if cached else None,
            "memory_note": "price reused from session cache" if cached else "no cache hit",
        }
        _session_memory[f"vol_{ticker}"] = result
        tracer.log("calculate_volatility",
                   {"ticker": ticker, "window": window}, result, time.time() - start)
        return json.dumps(result)
    except Exception as exc:
        err = {"error": str(exc)}
        tracer.log("calculate_volatility", {"ticker": ticker}, err, time.time() - start)
        return json.dumps(err)


@tool
def llm_sentiment(headlines_json: str) -> str:
    """
    Analyses financial news headline sentiment using Groq LLaMA-3.
    Returns per-headline sentiment (positive/negative/neutral), confidence, brief_reason,
    and an aggregate score. Output is Pydantic-validated.

    Args:
        headlines_json: JSON string — list of strings OR {'headlines': [...]} dict.
    """
    # AI-ASSISTED: Claude (claude-sonnet-4-6),
    # Prompt: 'Write a financial sentiment system prompt that returns structured JSON
    #          validated by Pydantic, with aggregate_score weighted by confidence',
    # Date: 2026-05-11
    SYSTEM = """You are a financial sentiment analyst. Given a list of news headlines,
return ONLY a raw JSON object (no markdown, no preamble):
{
  "analyses": [
    {"headline":"...","sentiment":"positive|negative|neutral","confidence":0.0,"brief_reason":"..."}
  ],
  "aggregate_score": 0.0,
  "overall_sentiment": "positive|negative|neutral"
}
Rules: confidence 0.0–1.0; aggregate_score = weighted avg of (confidence × polarity)
where positive=+1, negative=-1, neutral=0. Range −1 to +1."""

    start = time.time()
    llm   = ChatGroq(model=LLM_MODEL, temperature=0.0, max_tokens=2000)
    try:
        data = json.loads(headlines_json)
        lines = (
            [h if isinstance(h, str) else h.get("headline","") for h in data]
            if isinstance(data, list)
            else [h if isinstance(h, str) else h.get("headline","")
                  for h in data.get("headlines", [])]
            if isinstance(data, dict)
            else [str(data)]
        )
        text     = "\n".join(f"- {h}" for h in lines[:15])
        response = llm.invoke([SystemMessage(content=SYSTEM),
                                HumanMessage(content=f"Analyse:\n{text}")])
        raw = response.content.strip()
        if "```" in raw:
            raw = raw.split("```")[1].lstrip("json").strip()
        si = raw.find("{"); ei = raw.rfind("}") + 1
        raw = raw[si:ei] if si >= 0 else raw
        result    = json.loads(raw)
        validated = SentimentResult(**result)
        final     = validated.model_dump()
        _session_memory["sentiment"] = final
        tracer.log("llm_sentiment", {"n_headlines": len(lines)}, final, time.time() - start)
        return json.dumps(final)
    except (ValidationError, json.JSONDecodeError) as exc:
        err = {"error": f"Validation/parse error: {exc}",
               "analyses": [], "aggregate_score": 0.0, "overall_sentiment": "neutral"}
        tracer.log("llm_sentiment", {"input_len": len(headlines_json)}, err, time.time() - start)
        return json.dumps(err)
    except Exception as exc:
        err = {"error": str(exc)}
        tracer.log("llm_sentiment", {"input_len": len(headlines_json)}, err, time.time() - start)
        return json.dumps(err)


@tool
def web_search(query: str) -> str:
    """
    Searches the web for analyst commentary and financial news using DuckDuckGo.
    No API key required. Returns JSON with title, snippet, and URL for each result.

    Args:
        query: Search query string.
    """
    # AI-ASSISTED: Claude (claude-sonnet-4-6),
    # Prompt: 'Implement a robust duckduckgo-search tool wrapper with graceful fallback',
    # Date: 2026-05-11
    start = time.time()
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            raw = list(ddgs.text(query, max_results=5))
        results = [{"title": r.get("title",""), "snippet": r.get("body","")[:300],
                    "url": r.get("href","")} for r in raw]
        result  = {"query": query, "results": results, "count": len(results)}
        tracer.log("web_search", {"query": query}, result, time.time() - start)
        return json.dumps(result)
    except ImportError:
        err = {"error": "pip install duckduckgo-search", "results": []}
        tracer.log("web_search", {"query": query}, err, time.time() - start)
        return json.dumps(err)
    except Exception as exc:
        err = {"error": str(exc), "results": [],
               "fallback_hint": "Try a shorter, simpler query"}
        tracer.log("web_search", {"query": query}, err, time.time() - start)
        return json.dumps(err)


ALL_TOOLS     = [get_price_data, get_news, calculate_volatility, llm_sentiment, web_search]
AGENT_A_TOOLS = [get_price_data, calculate_volatility, llm_sentiment]
AGENT_B_TOOLS = [web_search, get_news]

print("✅ All 5 tools defined.")
print(f"   Agent A: {[t.name for t in AGENT_A_TOOLS]}")
print(f"   Agent B: {[t.name for t in AGENT_B_TOOLS]}")

✅ All 5 tools defined.
   Agent A: ['get_price_data', 'calculate_volatility', 'llm_sentiment']
   Agent B: ['web_search', 'get_news']


Step 7 — Task 3A: Single Research Agent

In [8]:
# STEP 7 — Task 3A: Single Research Agent (LangGraph ReAct)

def run_task_3a(ticker: str = DEFAULT_TICKER) -> str:
    """
    Task 3A: Single autonomous research agent via LangGraph create_react_agent.
    Agent decides tool order, observes each result, and replans autonomously.
    Uses all 5 tools. Produces 3-section structured final report.
    LangSmith traces every LLM call if LANGCHAIN_TRACING_V2 is set.
    """
    print("\n" + "=" * 66)
    print(f" TASK 3A — Single Research Agent  |  Ticker: {ticker}")
    print("=" * 66)

    llm = ChatGroq(model=LLM_MODEL, temperature=0.2, max_tokens=4096)

    # AI-ASSISTED: Claude (claude-sonnet-4-6),
    # Prompt: 'Write an observe-and-replan agent system prompt enforcing autonomous
    #          tool selection with graceful error handling and a 3-section report',
    # Date: 2026-05-11
    SYSTEM = f"""You are an expert autonomous equity research agent. Your mission:

QUERY: Analyse the current financial health and market sentiment of {ticker}.
       Identify the top three risks to its share price over the next 90 days
       and suggest one data-driven hedge strategy.

AUTONOMOUS BEHAVIOUR RULES:
1. After EACH tool result, reason explicitly: what did I learn? what should I do next?
2. Do NOT follow a fixed tool order — let the data guide your next action.
3. Use at least 4 of the 5 available tools.
4. If a tool returns an error, try an alternative query or a different tool.

MANDATORY FINAL REPORT FORMAT (after all tool calls):
---
## Financial Health Summary
[2–3 paragraphs: price, indicators, PE, YTD, sentiment]

## Top Three Risks
**Risk 1 — [Name]** (Severity: High|Medium|Low)
Evidence: [specific data from tools]

**Risk 2 — [Name]** (Severity: High|Medium|Low)
Evidence: ...

**Risk 3 — [Name]** (Severity: High|Medium|Low)
Evidence: ...

## Hedge Strategy Recommendation
Strategy: [specific strategy name]
Instruments: [specific instruments]
Rationale: [2–3 sentences referencing tool data]
---"""

    agent  = create_react_agent(model=llm, tools=ALL_TOOLS, prompt=SYSTEM)
    query  = (f"Analyse {ticker}: financial health, sentiment, top 3 risks "
              f"over next 90 days, and one data-driven hedge strategy.")

    print(f"\n🤖 Agent starting autonomous analysis of {ticker}...\n")
    result     = agent.invoke({"messages": [HumanMessage(content=query)]})
    tools_used = []
    final_text = ""

    for msg in result["messages"]:
        if isinstance(msg, AIMessage):
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"\n🔧 TOOL CALL → {tc['name']}")
                    print(f"   Args : {json.dumps(tc['args'])[:120]}")
                    tools_used.append(tc["name"])
            if msg.content:
                print(f"\n💭 Agent: {msg.content[:250]}...")
                final_text = msg.content
        elif isinstance(msg, ToolMessage):
            print(f"\n📊 [{msg.name}]: {str(msg.content)[:150]}...")

    print("\n" + "─" * 66)
    print("📋 TASK 3A — FINAL REPORT")
    print("─" * 66)
    print(final_text)
    print(f"\n✅ Tools invoked: {tools_used}")

    if LANGSMITH_ENABLED:
        print(f"🔗 LangSmith trace: https://smith.langchain.com — project: cdazzdev-task3-financial-research")

    _session_memory["task3a_report"] = final_text
    _session_memory["task3a_ticker"] = ticker
    return final_text

Step 8 — Task 3B: Multi-Agent Pipeline + Critique Loop

In [9]:
# STEP 8 — Task 3B: Multi-Agent Pipeline (Agent A + Agent B + Critique Loop)

import re
from groq import RateLimitError as _GroqRateLimitError

def _llm_invoke_with_retry(llm_with_tools, messages, tools_list=None, max_retries: int = 3):
    """
    Invoke the LLM with automatic model fallback + exponential back-off.

    Strategy:
      1. Try primary model (LLaMA-3.3-70b).
      2. On first RateLimitError → immediately switch to FALLBACK_MODEL
         (openai/gpt-4o-mini on Groq) and retry — no waiting needed.
      3. If fallback also rate-limited → wait then retry primary again.
      Repeats up to max_retries times before re-raising.
    """
    delays = [30, 60, 120, 180]
    current_llm = llm_with_tools
    used_fallback = False

    for attempt in range(1, max_retries + 2):
        try:
            return current_llm.invoke(messages)
        except _GroqRateLimitError as exc:
            err_str = str(exc)
            # ---- Try fallback model first (no wait) ----
            if not used_fallback and tools_list is not None:
                print(f"\n  🔄 LLaMA rate-limited → switching to fallback: {FALLBACK_MODEL}")
                try:
                    fallback_base = ChatGroq(model=FALLBACK_MODEL, temperature=0.2, max_tokens=4096)
                    current_llm   = fallback_base.bind_tools(tools_list)
                    used_fallback = True
                    return current_llm.invoke(messages)
                except _GroqRateLimitError:
                    print(f"  ⚠️  Fallback also rate-limited. Will wait and retry...")
                except Exception as fb_exc:
                    print(f"  ⚠️  Fallback error ({fb_exc}). Reverting to primary with wait...")
                    current_llm   = llm_with_tools   # revert
                    used_fallback = False
            # ---- Wait + retry ----
            match = re.search(r"try again in (\d+)m([\d.]+)s", err_str)
            wait  = (int(match.group(1)) * 60 + float(match.group(2)) + 5) if match \
                    else delays[min(attempt - 1, len(delays) - 1)]
            if attempt <= max_retries:
                model_label = FALLBACK_MODEL if used_fallback else LLM_MODEL
                print(f"\n  ⏳ Rate limit on {model_label} (attempt {attempt}/{max_retries}). "
                      f"Waiting {wait:.0f}s...")
                time.sleep(wait)
                current_llm = llm_with_tools   # retry primary after wait
                used_fallback = False
            else:
                raise


def _react_loop(llm_with_tools, system: str, tools_dict: dict,
                initial: str, name: str, max_iters: int = 8) -> tuple[str, list]:
    """
    Custom ReAct loop giving full control over message trace,
    enforced tool restriction, and critique loop detection.
    Falls back to FALLBACK_MODEL automatically via _llm_invoke_with_retry.
    """
    print(f"\n{'─' * 58}")
    print(f" {name}")
    print(f"{'─' * 58}")
    messages  = [SystemMessage(content=system), HumanMessage(content=initial)]
    tools_list = list(tools_dict.values())   # needed for fallback model binding

    for i in range(max_iters):
        response = _llm_invoke_with_retry(llm_with_tools, messages, tools_list=tools_list)
        messages.append(response)
        if response.content:
            print(f"\n💭 {name} [iter {i+1}]: {response.content[:220]}...")
        if not (hasattr(response, "tool_calls") and response.tool_calls):
            break
        for tc in response.tool_calls:
            t_name = tc["name"]; t_args = tc["args"]
            print(f"\n  🔧 {name} → {t_name}({json.dumps(t_args)[:80]})")
            t_result = (tools_dict[t_name].invoke(t_args)
                        if t_name in tools_dict
                        else json.dumps({"error": f"Tool '{t_name}' not available to {name}"}))
            print(f"  📊 Result: {str(t_result)[:100]}...")
            messages.append(ToolMessage(content=t_result,
                                        tool_call_id=tc["id"], name=t_name))
    return response.content, messages


def run_task_3b(ticker: str = DEFAULT_TICKER) -> tuple[str, Optional[FinalReport]]:
    """
    Task 3B: Two-agent pipeline with enforced tool separation, Pydantic handoff,
    and one critique loop cycle (Agent B → Agent A → Agent B).

    Agent A: get_price_data, calculate_volatility, llm_sentiment
    Agent B: web_search, get_news
    Handoff: DataBrief (Pydantic validated)
    Loop:    B sends clarification_request JSON → A responds → B finishes report
    """
    print("\n" + "=" * 66)
    print(f" TASK 3B — Multi-Agent Pipeline  |  Ticker: {ticker}")
    print("=" * 66)

    llm = ChatGroq(model=LLM_MODEL, temperature=0.2, max_tokens=4096)
    a_tools = {t.name: t for t in AGENT_A_TOOLS}
    b_tools = {t.name: t for t in AGENT_B_TOOLS}
    llm_a   = llm.bind_tools(AGENT_A_TOOLS)   # architecturally restricted
    llm_b   = llm.bind_tools(AGENT_B_TOOLS)   # architecturally restricted

    # ---- AGENT A ----
    # AI-ASSISTED: Claude (claude-sonnet-4-6),
    # Prompt: 'Write Agent A system prompt for quantitative equity data analysis
    #          with structured JSON DataBrief output',
    # Date: 2026-05-11
    A_SYS = f"""You are Agent A — quantitative equity data analyst.
AVAILABLE TOOLS: get_price_data, calculate_volatility, llm_sentiment.
DO NOT use web_search or get_news — they are not in your tool list.

Task: Complete quantitative analysis of {ticker}.
1. Call get_price_data first.
2. Call calculate_volatility.
3. If you have access to news headlines, call llm_sentiment; otherwise skip.

Output your final DataBrief as a JSON object:
{{
  "ticker": "{ticker}",
  "price_snapshot": {{...all price fields...}},
  "volatility_metrics": {{...all vol fields...}},
  "sentiment_result": {{...or empty dict if no headlines...}},
  "momentum_signal": "bullish|bearish|neutral",
  "key_observations": ["obs1","obs2","obs3"],
  "generated_at": "ISO8601"
}}
Return ONLY the JSON after your tool calls."""

    a_text, a_msgs = _react_loop(
        llm_a, A_SYS, a_tools,
        f"Perform full quantitative analysis of {ticker}.",
        "Agent A (Data Analyst)"
    )

    # ---- Parse & validate DataBrief ----
    print(f"\n📦 Validating Agent A → DataBrief (Pydantic) ...")
    try:
        raw = a_text.strip()
        if "```" in raw: raw = raw.split("```")[1].lstrip("json").strip()
        si = raw.find("{"); ei = raw.rfind("}") + 1
        a_json = json.loads(raw[si:ei]) if si >= 0 else {}
    except Exception as exc:
        print(f"  ⚠️  JSON parse warning ({exc}) — falling back to session memory.")
        a_json = {}

    try:
        brief = DataBrief(
            ticker            = a_json.get("ticker", ticker),
            price_snapshot    = a_json.get("price_snapshot")    or _session_memory.get(f"price_{ticker}", {}),
            volatility_metrics= a_json.get("volatility_metrics") or _session_memory.get(f"vol_{ticker}", {}),
            sentiment_result  = a_json.get("sentiment_result")  or _session_memory.get("sentiment", {}),
            momentum_signal   = a_json.get("momentum_signal", "neutral"),
            key_observations  = a_json.get("key_observations", []),
            generated_at      = a_json.get("generated_at", datetime.utcnow().isoformat()),
        )
        print(f"  ✅ DataBrief validated.")
    except ValidationError as exc:
        print(f"  ⚠️  Pydantic validation note: {exc} — using session memory fallback.")
        brief = DataBrief(
            ticker=ticker,
            price_snapshot=_session_memory.get(f"price_{ticker}", {}),
            volatility_metrics=_session_memory.get(f"vol_{ticker}", {}),
            sentiment_result=_session_memory.get("sentiment", {}),
            momentum_signal="neutral",
            generated_at=datetime.utcnow().isoformat(),
        )

    db = brief.model_dump()
    print("\n" + "═" * 58)
    print("  STRUCTURED HANDOFF: Agent A → Agent B  (DataBrief)")
    print("═" * 58)
    print(f"  Ticker     : {db['ticker']}")
    print(f"  Price      : ${db['price_snapshot'].get('current_price','N/A')}")
    print(f"  Momentum   : {db['momentum_signal']}")
    print(f"  Vol Regime : {db['volatility_metrics'].get('vol_regime','N/A')}")
    print(f"  Beta(SPY)  : {db['volatility_metrics'].get('beta_vs_spy','N/A')}")
    print(f"  Sentiment  : {db['sentiment_result'].get('overall_sentiment','N/A')}")
    print(f"  Schema     : Pydantic DataBrief — typed, validated")
    print("═" * 58)

    # ---- AGENT B — Phase 1 ----
    # AI-ASSISTED: Claude (claude-sonnet-4-6),
    # Prompt: 'Write Agent B system prompt for qualitative research synthesis
    #          with structured clarification_request critique loop JSON',
    # Date: 2026-05-11
    B_SYS = f"""You are Agent B — senior equity research writer.
AVAILABLE TOOLS: web_search, get_news.
DO NOT call get_price_data or calculate_volatility — use only the DataBrief from Agent A.

Workflow:
1. Call get_news for {ticker} headlines.
2. Call web_search for analyst opinions and risk factors.
3. Evaluate: do you need additional quantitative detail from Agent A?
   If YES — output ONLY this JSON (nothing else):
   {{"clarification_request":"EXACT DATA NEEDED","reason":"WHY ESSENTIAL"}}
   If NO — write the full final report directly.

FINAL REPORT FORMAT:
## Financial Health Summary
[2 paragraphs integrating DataBrief data + web/news context]

## Top Three Risks
**Risk 1 — [Name]** (Severity: High|Medium|Low)
Evidence: [cite DataBrief fields AND/OR news/web sources]

**Risk 2 — [Name]** (Severity: High|Medium|Low)
Evidence: ...

**Risk 3 — [Name]** (Severity: High|Medium|Low)
Evidence: ...

## Hedge Strategy Recommendation
Strategy: [specific name]
Instruments: [list specific instruments]
Rationale: [2–3 sentences citing vol={db['volatility_metrics'].get('annual_vol_pct','?')}%,
beta={db['volatility_metrics'].get('beta_vs_spy','?')}, momentum={db['momentum_signal']}]"""

    brief_json = json.dumps(db, indent=2)
    b_init = (f"DataBrief from Agent A for {ticker}:\n\n{brief_json}\n\n"
              "Gather qualitative context with your tools. Then either send a "
              "clarification_request or write the final report.")

    b_text, b_msgs = _react_loop(
        llm_b, B_SYS, b_tools, b_init,
        "Agent B (Research Writer) — Phase 1"
    )

    # ---- CRITIQUE LOOP ----
    print("\n" + "═" * 58)
    print("  CRITIQUE LOOP: Agent B ↔ Agent A")
    print("═" * 58)
    clarification_data = ""

    if "clarification_request" in b_text:
        print("  🔄 Agent B sent a clarification request to Agent A!")
        try:
            raw = b_text.strip()
            si  = raw.find("{"); ei = raw.rfind("}") + 1
            clr = json.loads(raw[si:ei]) if si >= 0 else {}
            req = clr.get("clarification_request", b_text[:200])
            rsn = clr.get("reason", "")
            print(f"  ❓ Request : {req}")
            print(f"  📌 Reason  : {rsn}")
        except Exception:
            req = b_text[:200]

        # Agent A answers from already-fetched session data
        print("\n  🤖 Agent A responding (from session data — no new tool calls) ...")
        llm_plain   = ChatGroq(model=LLM_MODEL, temperature=0.0, max_tokens=800)
        a_clarify   = llm_plain.invoke([
            SystemMessage(content=(
                f"You are Agent A. Answer the clarification request about {ticker} "
                "using ONLY data you've already retrieved. Be specific and factual. "
                "DO NOT call any tools."
            )),
            HumanMessage(content=(
                f"Clarification requested: {req}\n\n"
                f"Your retrieved data:\n{json.dumps(db, indent=2)}"
            )),
        ])
        clarification_data = a_clarify.content
        print(f"  ✅ Agent A answer: {clarification_data[:260]}...")

        # Agent B incorporates and writes final report
        print("\n  📝 Agent B incorporating clarification → final report ...")
        b_msgs.append(HumanMessage(content=(
            f"Agent A clarification:\n{clarification_data}\n\n"
            "Now synthesise all gathered information and write the full final report."
        )))
        llm_b2 = ChatGroq(model=LLM_MODEL, temperature=0.2, max_tokens=3000)
        final_r = llm_b2.invoke(b_msgs)
        b_msgs.append(final_r)
        b_text = final_r.content
        print("  ✅ Critique loop complete — Agent B incorporated Agent A's response.")
    else:
        print("  ℹ️  Agent B produced report directly (no clarification needed).")

    # ---- Final report ----
    print("\n" + "─" * 66)
    print("📋 TASK 3B — FINAL RESEARCH REPORT")
    print("─" * 66)
    print(b_text)

    _session_memory["task3b_report"]     = b_text
    _session_memory["task3b_data_brief"] = db
    _session_memory["task3b_critique"]   = clarification_data

    # Pydantic FinalReport validation
    try:
        final_report = FinalReport(
            ticker                   = ticker,
            financial_health_summary = b_text[:600],
            top_three_risks          = [
                RiskItem(risk="See full report above",
                         supporting_evidence="Integrated from Agent A DataBrief + Agent B research",
                         severity="high")
            ],
            hedge_strategy = HedgeStrategy(
                strategy    = "See Hedge Strategy section in report",
                rationale   = f"vol={db['volatility_metrics'].get('annual_vol_pct','?')}%, "
                              f"momentum={db['momentum_signal']}",
                instruments = ["Options", "Inverse ETF", "SPY hedge"],
            ),
            sources_consulted = ["yfinance", "DuckDuckGo web_search", "Groq LLaMA-3"],
            generated_at      = datetime.utcnow().isoformat(),
        )
        print(f"\n  ✅ FinalReport Pydantic model validated.")
    except ValidationError as exc:
        print(f"  ⚠️  FinalReport validation note: {exc}")
        final_report = None

    if LANGSMITH_ENABLED:
        print(f"\n🔗 LangSmith: https://smith.langchain.com — project: cdazzdev-task3-financial-research")

    return b_text, final_report

Step 9 — Task 3C: Memory & Observability

In [10]:
# STEP 9 — Task 3C: Memory & Observability

def demonstrate_short_term_memory(ticker: str = DEFAULT_TICKER) -> str:
    """
    Short-term memory demo: answers a follow-up question from session cache
    WITHOUT re-calling any tool. Demonstrates context carried across tool calls.
    """
    print("\n" + "=" * 66)
    print(f" TASK 3C — Short-Term Memory Demo  |  Ticker: {ticker}")
    print("=" * 66)
    price = _session_memory.get(f"price_{ticker}")
    vol   = _session_memory.get(f"vol_{ticker}")
    sent  = _session_memory.get("sentiment")

    if not price:
        print("  ⚠️  Session memory empty — run 3A or 3B first.")
        return ""

    print("\n  📌 Session memory contents:")
    print(f"    price_{ticker}  → price=${price.get('current_price')}, RSI={price.get('rsi_14')}, momentum={price.get('momentum_signal')}")
    if vol:  print(f"    vol_{ticker}    → {vol.get('annual_vol_pct','?')}% vol, regime={vol.get('vol_regime')}, beta={vol.get('beta_vs_spy')}")
    if sent: print(f"    sentiment       → {sent.get('overall_sentiment','?')}, score={sent.get('aggregate_score','?')}")

    followup = (f"Based on the data already retrieved for {ticker}, is the stock "
                "currently overbought or oversold, and what does the volatility regime "
                "suggest about near-term downside risk?")
    print(f"\n  ❓ Follow-up question (ZERO tool calls will be made):")
    print(f"     {followup}")

    llm     = ChatGroq(model=LLM_MODEL, temperature=0.1, max_tokens=400)
    context = (
        f"Use ONLY this cached session data to answer the follow-up question. "
        f"Do NOT call any tools.\n\n"
        f"Price: {json.dumps(price)}\n"
        f"Vol  : {json.dumps(vol) if vol else 'not in cache'}\n"
        f"Sent : {json.dumps(sent) if sent else 'not in cache'}"
    )
    answer = llm.invoke([SystemMessage(content=context),
                          HumanMessage(content=followup)]).content
    print(f"\n  💡 Answer (from session memory, no tool calls):\n  {answer}")
    print("\n  ✅ Short-term memory verified.")
    return answer


def save_persistent_cache(ticker: str) -> Path:
    """Saves the session's research brief to research_cache/{ticker}_{date}.json."""
    date_key   = datetime.now().strftime("%Y-%m-%d")
    cache_file = CACHE_DIR / f"{ticker}_{date_key}.json"
    report     = (_session_memory.get("task3b_report")
                  or _session_memory.get("task3a_report")
                  or "No report generated.")
    payload = {
        "ticker":      ticker,
        "date":        date_key,
        "report":      report,
        "data_brief":  _session_memory.get("task3b_data_brief", {}),
        "cached_at":   datetime.utcnow().isoformat(),
    }
    with open(cache_file, "w") as fh:
        json.dump(payload, fh, indent=2, default=str)
    print(f"\n💾 Persistent cache saved → {cache_file}")
    return cache_file


def load_persistent_cache(ticker: str) -> Optional[dict]:
    """
    Detects a cached brief for today's date.
    Returns cached dict (skip all tools) or None if not found.
    """
    date_key   = datetime.now().strftime("%Y-%m-%d")
    cache_file = CACHE_DIR / f"{ticker}_{date_key}.json"
    if cache_file.exists():
        with open(cache_file) as fh:
            data = json.load(fh)
        print(f"✅ CACHE HIT: {cache_file.name} — skipping all tool calls.")
        print(f"   Cached at: {data.get('cached_at')}")
        print(f"   Report   : {str(data.get('report',''))[:200]}...")
        return data
    print(f"ℹ️  CACHE MISS: No cache for {ticker} on {date_key}. Running full pipeline...")
    return None


def print_trace_summary():
    """Prints a formatted summary of agent_trace.jsonl to notebook output."""
    print("\n" + "=" * 66)
    print(f" OBSERVABILITY — agent_trace.jsonl Summary")
    print("=" * 66)
    if not Path(TRACE_FILE).exists():
        print(f"  ⚠️  {TRACE_FILE} not found."); return
    records = []
    with open(TRACE_FILE) as fh:
        for line in fh:
            line = line.strip()
            if line:
                try: records.append(json.loads(line))
                except Exception: pass
    sessions = list(dict.fromkeys(r["session_id"] for r in records))
    print(f"\n  Total tool calls : {len(records)}")
    print(f"  Sessions         : {sessions}")
    print(f"\n  {'Tool':<28} {'Duration':>9}  {'Output (truncated)'}")
    print(f"  {'─'*28} {'─'*9}  {'─'*30}")
    for r in records:
        out = r["output_truncated"].replace("\n", " ")[:38]
        print(f"  {r['tool_name']:<28} {r['duration_seconds']:>8.2f}s  {out}...")
    print(f"\n  ✅ Commit {TRACE_FILE} → task3_agentic/logs/")

Step 10 — Main Runner

In [11]:
# STEP 10 — Main Runner (execute this cell)

TICKER = DEFAULT_TICKER   # ← change to any ticker

print(f"\n{'#'*66}")
print(f"# CDAZZDEV-MLE Task 3 — Multi-Agent Financial Research")
print(f"# Ticker: {TICKER} | LangSmith: {'ON' if LANGSMITH_ENABLED else 'OFF'}")
print(f"{'#'*66}\n")

# ---- Persistent cache check ----
cached = load_persistent_cache(TICKER)

if cached:
    print("🎉 Using cached result. Delete cache file and re-run to force refresh.")
    print("─" * 66)
    print(cached.get("report", "")[:800])
else:
    # ---- Task 3A ----
    report_3a = run_task_3a(TICKER)

    # ---- Task 3B ----
    print("\n  ⏳ Pausing 3 min between Task 3A and 3B to respect Groq daily token limits...")
    time.sleep(180)   # 3-minute pause — free tier TPD limit is 100k tokens
    report_3b_text, report_3b_obj = run_task_3b(TICKER)

    # ---- Task 3C: Short-term memory ----
    _ = demonstrate_short_term_memory(TICKER)

    # ---- Task 3C: Persistent cache ----
    print("\n" + "=" * 66)
    print(f" TASK 3C — Saving Persistent Cache")
    print("=" * 66)
    cache_file = save_persistent_cache(TICKER)
    print(f"\n  ✅ Saved. Next run will detect {cache_file.name} and skip all tool calls.")

# ---- Trace summary ----
print_trace_summary()

print("\n" + "=" * 66)
print("  ✅  TASK 3A + 3B + 3C COMPLETE")
print("=" * 66)
trace_size = Path(TRACE_FILE).stat().st_size if Path(TRACE_FILE).exists() else 0
print(f"  agent_trace.jsonl : {trace_size} bytes")
print(f"  Cache files       : {[f.name for f in CACHE_DIR.iterdir()]}")



##################################################################
# CDAZZDEV-MLE Task 3 — Multi-Agent Financial Research
# Ticker: NVDA | LangSmith: ON
##################################################################

ℹ️  CACHE MISS: No cache for NVDA on 2026-05-11. Running full pipeline...

 TASK 3A — Single Research Agent  |  Ticker: NVDA

🤖 Agent starting autonomous analysis of NVDA...

  📋 [get_price_data] 0.35s → {'ticker': 'NVDA', 'current_price': 215.2, '52w_high': ...
  📋 [get_news] 0.39s → {'ticker': 'NVDA', 'headlines': [{'headline': "Tech sto...


/tmp/ipykernel_11730/3376901062.py:251: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


  📋 [calculate_volatility] 0.39s → {'ticker': 'NVDA', 'window_days': 90, 'annual_vol_pct':...
  📋 [web_search] 0.41s → {'query': 'NVDA analyst commentary', 'results': [], 'co...


sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7cc178778130>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7cc17c241c50>


  📋 [llm_sentiment] 14.03s → {'analyses': [{'headline': "Tech stocks today: AI chipm...


...

/tmp/ipykernel_11730/3376901062.py:251: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



💭 Agent B (Research Writer) — Phase 1 [iter 2]: ## Financial Health Summary
NVIDIA's (NVDA) current price is above its 50-day and 200-day moving averages, indicating a bullish trend. The stock's RSI-14 is 65.86, suggesting a slightly overbought condition. According to...

══════════════════════════════════════════════════════════
  CRITIQUE LOOP: Agent B ↔ Agent A
══════════════════════════════════════════════════════════
  ℹ️  Agent B produced report directly (no clarification needed).

──────────────────────────────────────────────────────────────────
📋 TASK 3B — FINAL RESEARCH REPORT
──────────────────────────────────────────────────────────────────
## Financial Health Summary
NVIDIA's (NVDA) current price is above its 50-day and 200-day moving averages, indicating a bullish trend. The stock's RSI-14 is 65.86, suggesting a slightly overbought condition. According to recent news headlines, tech stocks, including NVDA, have been performing well, with some analysts predicting a potent

/tmp/ipykernel_11730/1931974486.py:309: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  generated_at      = datetime.utcnow().isoformat(),



  💡 Answer (from session memory, no tool calls):
  Based on the data, the Relative Strength Index (RSI_14) is 65.86, which is not in the overbought range (typically above 70). This suggests that the stock is not currently overbought.

The volatility regime is classified as "normal" with an annual volatility percentage of 36.9. This suggests that the near-term downside risk is not unusually high. However, the max daily loss percentage is -5.61, which indicates that the stock has experienced significant daily losses in the past. 

Overall, the data does not indicate that the stock is currently overbought, and the volatility regime suggests a normal level of near-term downside risk.

  ✅ Short-term memory verified.

 TASK 3C — Saving Persistent Cache

💾 Persistent cache saved → research_cache/NVDA_2026-05-11.json

  ✅ Saved. Next run will detect NVDA_2026-05-11.json and skip all tool calls.

 OBSERVABILITY — agent_trace.jsonl Summary

  Total tool calls : 15
  Sessions         : ['4f9f23

/tmp/ipykernel_11730/2496498721.py:57: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "cached_at":   datetime.utcnow().isoformat(),


Step 11 — Write Streamlit Dashboard File

In [12]:
# STEP 11 — Write Streamlit Dashboard File
# AI-ASSISTED: Claude (claude-sonnet-4-6),
# Prompt: 'Build a Streamlit dashboard that reads agent_trace.jsonl and displays
#          tool call timeline, duration charts, session selector, and message trace',
# Date: 2026-05-11

DASHBOARD_CODE = '''
import streamlit as st
import json, re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime

st.set_page_config(
    page_title="Agent Trace — CDAZZDEV Task 3",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ---- Custom CSS ----
st.markdown("""
<style>
.metric-card {
    background: #f8f9fa;
    border-left: 4px solid #4f46e5;
    padding: 12px 16px;
    border-radius: 6px;
    margin-bottom: 8px;
}
.tool-pill {
    display: inline-block;
    padding: 2px 10px;
    border-radius: 12px;
    font-size: 12px;
    font-weight: 600;
    margin: 2px;
}
.pill-price { background:#dbeafe; color:#1e40af; }
.pill-vol   { background:#fce7f3; color:#9d174d; }
.pill-news  { background:#d1fae5; color:#065f46; }
.pill-sent  { background:#fef3c7; color:#92400e; }
.pill-web   { background:#ede9fe; color:#4c1d95; }
.badge-ok  { background:#d1fae5; color:#065f46; padding:2px 8px; border-radius:10px; font-size:11px; }
.badge-err { background:#fee2e2; color:#991b1b; padding:2px 8px; border-radius:10px; font-size:11px; }
</style>
""", unsafe_allow_html=True)

PILL = {
    "get_price_data":       "pill-price",
    "calculate_volatility": "pill-vol",
    "get_news":             "pill-news",
    "llm_sentiment":        "pill-sent",
    "web_search":           "pill-web",
}

# ---- Load trace ----
TRACE_FILE = "agent_trace.jsonl"

@st.cache_data(ttl=5)
def load_trace():
    records = []
    p = Path(TRACE_FILE)
    if p.exists():
        with open(p) as f:
            for line in f:
                line = line.strip()
                if line:
                    try: records.append(json.loads(line))
                    except: pass
    return records

@st.cache_data(ttl=5)
def load_cache_files():
    files = []
    for p in Path("research_cache").glob("*.json"):
        try:
            with open(p) as f:
                data = json.load(f)
            files.append({"file": p.name, "ticker": data.get("ticker","?"),
                           "date": data.get("date","?"),
                           "report_preview": str(data.get("report",""))[:120]})
        except: pass
    return files

records = load_trace()

# ---- Sidebar ----
with st.sidebar:
    st.image("https://img.shields.io/badge/CDAZZDEV-MLE-4f46e5?style=for-the-badge", width=200)
    st.markdown("### 🤖 Agent Trace Dashboard")
    st.caption("Task 3 — Multi-Agent Financial Research")
    st.divider()

    if not records:
        st.error("No agent_trace.jsonl found.\\nRun the notebook first, then refresh.")
        st.stop()

    sessions = ["All"] + list(dict.fromkeys(r["session_id"] for r in records))
    sel_session = st.selectbox("Session", sessions, index=0)
    st.divider()

    tools_all = list(dict.fromkeys(r["tool_name"] for r in records))
    sel_tools = st.multiselect("Filter tools", tools_all, default=tools_all)
    st.divider()

    st.caption(f"📄 Trace file: `{TRACE_FILE}`")
    if st.button("🔄 Refresh data"):
        st.cache_data.clear()
        st.rerun()

# ---- Filter ----
df = pd.DataFrame(records)
print(df.columns.tolist())
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["has_error"]  = df["output_truncated"].str.lower().str.contains("error", na=False)

if sel_session != "All":
    df = df[df["session_id"] == sel_session]
df = df[df["tool_name"].isin(sel_tools)]

# ---- Header ----
st.title("🔍 Agent Trace Dashboard")
st.caption(
    f"CDAZZDEV-MLE Task 3 · {len(df)} tool calls · "
    f"{df['session_id'].nunique()} session(s) · "
    f"Last call: {df['timestamp'].max().strftime('%H:%M:%S') if len(df) > 0 else 'N/A'}"
)

# ---- KPI Row ----
c1, c2, c3, c4, c5 = st.columns(5)
c1.metric("Total Calls",     len(df))
c2.metric("Avg Duration",    f"{df['duration_seconds'].mean():.2f}s")
c3.metric("Slowest Call",    f"{df['duration_seconds'].max():.2f}s")
c4.metric("Error Calls",     int(df['has_error'].sum()))
c5.metric("Unique Tools",    df['tool_name'].nunique())

st.divider()

# ---- Two-column layout ----
col_left, col_right = st.columns([3, 2])

with col_left:
    st.subheader("📊 Tool Call Duration")
    fig_bar = px.bar(
        df.sort_values("timestamp"),
        x="tool_name", y="duration_seconds",
        color="tool_name",
        text="duration_seconds",
        labels={"tool_name": "Tool", "duration_seconds": "Duration (s)"},
        color_discrete_map={
            "get_price_data": "#3b82f6",
            "calculate_volatility": "#ec4899",
            "get_news": "#10b981",
            "llm_sentiment": "#f59e0b",
            "web_search": "#8b5cf6",
        },
        template="plotly_white",
    )
    fig_bar.update_traces(texttemplate="%{text:.2f}s", textposition="outside")
    fig_bar.update_layout(showlegend=False, height=320,
                           margin=dict(l=0, r=0, t=20, b=40),
                           xaxis_title="", yaxis_title="seconds")
    st.plotly_chart(fig_bar, use_container_width=True)

with col_right:
    st.subheader("🥧 Tool Distribution")
    counts = df["tool_name"].value_counts().reset_index()
    counts.columns = ["tool", "count"]
    fig_pie = px.pie(
        counts, names="tool", values="count",
        color="tool",
        color_discrete_map={
            "get_price_data": "#3b82f6", "calculate_volatility": "#ec4899",
            "get_news": "#10b981", "llm_sentiment": "#f59e0b", "web_search": "#8b5cf6",
        },
        hole=0.45, template="plotly_white",
    )
    fig_pie.update_traces(textinfo="percent+label")
    fig_pie.update_layout(showlegend=False, height=320,
                           margin=dict(l=0, r=0, t=20, b=0))
    st.plotly_chart(fig_pie, use_container_width=True)

st.divider()

# ---- Timeline ----
st.subheader("⏱️ Tool Call Timeline")
if df["timestamp"].notna().any():
    df_sorted = df.sort_values("timestamp").reset_index(drop=True)
    df_sorted["call_index"] = range(len(df_sorted))
    df_sorted["end_time"]   = df_sorted["timestamp"] + pd.to_timedelta(df_sorted["duration_seconds"], unit="s")

    fig_tl = px.timeline(
        df_sorted,
        x_start="timestamp", x_end="end_time",
        y="tool_name", color="tool_name",
        hover_data=["duration_seconds", "session_id"],
        color_discrete_map={
            "get_price_data": "#3b82f6", "calculate_volatility": "#ec4899",
            "get_news": "#10b981", "llm_sentiment": "#f59e0b", "web_search": "#8b5cf6",
        },
        template="plotly_white",
        labels={"tool_name": "Tool"},
    )
    fig_tl.update_yaxes(autorange="reversed")
    fig_tl.update_layout(showlegend=False, height=280,
                          margin=dict(l=0, r=0, t=10, b=30))
    st.plotly_chart(fig_tl, use_container_width=True)
else:
    st.info("Timestamps not available for timeline view.")

st.divider()

# ---- Detailed trace table ----
st.subheader("📋 Detailed Trace Log")

for i, row in df.iterrows():
    pill_cls = PILL.get(row["tool_name"], "pill-price")
    status   = "🔴 Error" if row["has_error"] else "✅ OK"
    ts_str   = row["timestamp"].strftime("%H:%M:%S.%f")[:-3] if pd.notna(row["timestamp"]) else "N/A"
    label    = (f"<span class='tool-pill {pill_cls}'>{row['tool_name']}</span>&nbsp;"
                f"<small>{ts_str}</small>&nbsp;"
                f"<small>{row['duration_seconds']:.2f}s</small>&nbsp;"
                f"<small>{status}</small>")
    with st.expander(row["tool_name"] + f"  —  {ts_str}  —  {row['duration_seconds']:.2f}s  {status}"):
        ic, oc = st.columns(2)
        with ic:
            st.markdown("**Inputs**")
            try:
                inp = row["inputs"] if isinstance(row["inputs"], dict) else json.loads(row["inputs"])
                st.json(inp)
            except:
                st.code(str(row["inputs"]))
        with oc:
            st.markdown("**Output (truncated to 200 chars)**")
            raw_out = row["output_truncated"]
            try:
                out_parsed = json.loads(raw_out)
                st.json(out_parsed)
            except:
                st.code(raw_out)
        st.caption(f"Session: `{row['session_id']}`")

st.divider()

# ---- Persistent cache viewer ----
st.subheader("💾 Persistent Research Cache")
cache_files = load_cache_files()
if cache_files:
    for cf in cache_files:
        with st.expander(f"📄 {cf['file']}  |  {cf['ticker']}  |  {cf['date']}"):
            st.caption(cf["report_preview"] + "...")
else:
    st.info("No cache files found. Run the full pipeline to generate them.")

st.divider()

# ---- Session stats ----
st.subheader("📈 Per-Session Statistics")
if df["session_id"].nunique() > 0:
    sess_df = (df.groupby("session_id")
                 .agg(calls=("tool_name","count"),
                      total_time=("duration_seconds","sum"),
                      avg_time=("duration_seconds","mean"),
                      errors=("has_error","sum"))
                 .reset_index()
                 .round(3))
    st.dataframe(sess_df, use_container_width=True)

st.divider()
st.caption("CDAZZDEV-MLE Task 3 | Multi-Agent Financial Research System | Observability Dashboard")
'''

with open(DASHBOARD_FILE, "w") as fh:
    fh.write(DASHBOARD_CODE)

print(f"✅ dashboard.py written ({DASHBOARD_FILE.stat().st_size} bytes)")
print(f"   Ready to launch in Cell 12.")

✅ dashboard.py written (9179 bytes)
   Ready to launch in Cell 12.


Step 12 — Launch Streamlit Dashboard in Colab

In [13]:
# STEP 12 — BONUS: Launch Streamlit Dashboard in Colab
"""
Run this cell to launch the Streamlit dashboard.
It tries three methods in order — the first that works gives you a URL.

HOW TO TAKE THE SCREENSHOT FOR YOUR README:
1. Run this cell → copy the public URL printed below
2. Open the URL in your browser
3. Screenshot the full page (all tabs visible)
4. Save as:  task3_agentic/screenshots/dashboard_screenshot.png
5. Add to README.md:  ![Dashboard](task3_agentic/screenshots/dashboard_screenshot.png)
"""

import subprocess, threading, time, os

# ---- Kill any existing Streamlit processes ----
os.system("pkill -f streamlit 2>/dev/null || true")
time.sleep(1)

# Start Streamlit in the background
proc = subprocess.Popen(
    ["streamlit", "run", "dashboard.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false",
     "--server.enableXsrfProtection=false"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)
print("✅ Streamlit started on port 8501")

# ---- Method 1: Google Colab native proxy (no extra token) ----
try:
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(8501)")
    print(f"\n🚀 Dashboard URL (Colab proxy):\n   {url}")
    print("\n📸 Open this URL → take a screenshot → save to task3_agentic/screenshots/")
except Exception as e1:
    print(f"  Method 1 (Colab proxy) failed: {e1}")

    # ---- Method 2: pyngrok (requires free ngrok account) ----
    try:
        from pyngrok import ngrok, conf
        NGROK_TOKEN = ""
        try:
            from google.colab import userdata as _ud2
            NGROK_TOKEN = _ud2.get("NGROK_AUTHTOKEN") or ""
        except Exception: pass
        if not NGROK_TOKEN:
            NGROK_TOKEN = input("🔑 ngrok auth token (free at ngrok.com, or press Enter to skip): ").strip()
        if NGROK_TOKEN:
            ngrok.kill()
            conf.get_default().auth_token = NGROK_TOKEN
            tunnel = ngrok.connect(8501)
            print(f"\n🚀 Dashboard URL (ngrok):\n   {tunnel.public_url}")
            print("\n📸 Open this URL → take a screenshot → save to task3_agentic/screenshots/")
        else:
            raise ValueError("No ngrok token")
    except Exception as e2:
        print(f"  Method 2 (pyngrok) failed: {e2}")

        # ---- Method 3: localtunnel (no account needed) ----
        try:
            os.system("npm install -g localtunnel 2>/dev/null")
            lt_proc = subprocess.Popen(
                ["npx", "localtunnel", "--port", "8501"],
                stdout=subprocess.PIPE, stderr=subprocess.DEVNULL
            )
            for _ in range(15):
                line = lt_proc.stdout.readline().decode().strip()
                if "https://" in line:
                    print(f"\n🚀 Dashboard URL (localtunnel):\n   {line}")
                    print("   Note: localtunnel may ask for a password at first visit.")
                    print("   Password = your Colab's external IP (run: !curl ifconfig.me)")
                    print("\n📸 Open this URL → take a screenshot → save to task3_agentic/screenshots/")
                    break
                time.sleep(1)
        except Exception as e3:
            print(f"  Method 3 (localtunnel) failed: {e3}")
            print("\n⚠️  Could not expose dashboard. Download agent_trace.jsonl")
            print("   and run locally:  streamlit run dashboard.py")

print("\n💡 Dashboard reads agent_trace.jsonl automatically.")
print("   Refresh browser after running more pipeline cells.")
print("   Stop dashboard: proc.terminate()")

✅ Streamlit started on port 8501

🚀 Dashboard URL (Colab proxy):
   https://8501-gpu-t4-s-kkb-usw1b2-3qkuypvjj7liy-b.us-west1-2.prod.colab.dev

📸 Open this URL → take a screenshot → save to task3_agentic/screenshots/

💡 Dashboard reads agent_trace.jsonl automatically.
   Refresh browser after running more pipeline cells.
   Stop dashboard: proc.terminate()


Step 13 — LangSmith Screenshot Instructions

In [14]:
# STEP 13 — LangSmith Screenshot Instructions
"""
IF you enabled LangSmith in Cell 2, your traces are already uploading.

To get the LangSmith screenshot for your README:

1. Go to https://smith.langchain.com
2. Click "Projects" in the left sidebar
3. Open project: cdazzdev-task3-financial-research
4. You will see all agent runs with full tool call traces
5. Click on any run to see the detailed trace
6. Screenshot the trace view
7. Save as: task3_agentic/screenshots/langsmith_trace.png
8. Add to README.md:
   ![LangSmith Trace](task3_agentic/screenshots/langsmith_trace.png)

The LangSmith trace captures:
- Every LLM call with inputs and outputs
- Tool invocations with arguments and results
- Latency per step
- Token usage per call
- The full agent ReAct loop
"""
print("✅ See Cell 13 docstring for LangSmith screenshot instructions.")

✅ See Cell 13 docstring for LangSmith screenshot instructions.


Step 14 — Download Files for Submission

In [15]:
# STEP 14 — Download Files for Submission
"""
Run this to download all submission files to your local machine.
"""
from google.colab import files as colab_files
import zipfile

# Create submission zip
with zipfile.ZipFile("task3_submission.zip", "w") as zf:
    if Path(TRACE_FILE).exists():
        zf.write(TRACE_FILE, f"task3_agentic/logs/{TRACE_FILE}")
    if DASHBOARD_FILE.exists():
        zf.write(str(DASHBOARD_FILE), f"task3_agentic/{DASHBOARD_FILE}")
    for f in CACHE_DIR.iterdir():
        zf.write(str(f), f"task3_agentic/research_cache/{f.name}")

print("✅ task3_submission.zip created")
print("   Contents:")
with zipfile.ZipFile("task3_submission.zip") as zf:
    for name in zf.namelist():
        print(f"   {name}")

# Download individual files
colab_files.download(TRACE_FILE)
colab_files.download(str(DASHBOARD_FILE))
colab_files.download("task3_submission.zip")
print("\n✅ Downloads initiated.")

✅ task3_submission.zip created
   Contents:
   task3_agentic/logs/agent_trace.jsonl
   task3_agentic/dashboard.py
   task3_agentic/research_cache/NVDA_2026-05-11.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Downloads initiated.
